# CIPHER — Kaggle Benchmark Notebook
**Calibrated Introspection via Partially Hidden Environment Rules**

Dataset attached at `/kaggle/input/datasets/wenhaolu49/cipher`.

| Dimension | Weight | What it measures |
|-----------|--------|------------------|
| Objective | 35% | Plan quality vs. oracle beam search |
| Calibration | 25% | Brier score on stated self-knowledge |
| Attention | 20% | Rank correlation on unknown importance |
| Executive | 20% | Plan structure, named risks, alternative plans |

In [ ]:
# 1. Install dependencies
import subprocess, sys

def _pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", *packages], check=True)

_pip("kaggle-benchmarks")
_pip("google-genai")
_pip("tqdm")
print("Dependencies installed.")

In [ ]:
# 2. Imports and path setup
import os, sys, json, random
from typing import Any, Dict, List, Optional

_KAGGLE_ROOT = "/kaggle/input/datasets/wenhaolu49/cipher"
_LOCAL_ROOT  = os.path.abspath(os.getcwd())
_LOCAL_PARENT= os.path.abspath(os.path.join(os.getcwd(), ".."))

CIPHER_ROOT = None
for _c in [_KAGGLE_ROOT, _LOCAL_ROOT, _LOCAL_PARENT]:
    if os.path.isdir(os.path.join(_c, "cipher")):
        CIPHER_ROOT = _c
        break
if CIPHER_ROOT is None:
    raise RuntimeError("cipher/ package not found — attach the cipher dataset.")
if CIPHER_ROOT not in sys.path:
    sys.path.insert(0, CIPHER_ROOT)
print(f"cipher root: {CIPHER_ROOT}")

from cipher import build_prompt
from cipher.generator import Instance
from cipher.world import World, State, EntityState, Rule, Trigger, Effect, Action
from cipher.schema import validate_response
from cipher.scorer import score_response
from cipher.optimal import oracle_score
print("cipher imports OK.")

In [ ]:
# 3. Configuration
DATA_PATH = next(
    (p for p in [
        os.path.join(_KAGGLE_ROOT, "data", "instances.jsonl"),
        os.path.join(CIPHER_ROOT,  "data", "instances.jsonl"),
    ] if os.path.exists(p)),
    None,
)
if DATA_PATH is None:
    raise FileNotFoundError("data/instances.jsonl not found.")

# Set to None for the full 1000-instance submission run
KBENCH_LIMIT = 50
print(f"DATA_PATH    : {DATA_PATH}")
print(f"KBENCH_LIMIT : {KBENCH_LIMIT}")

In [ ]:
# 4. Load instances
def _load_jsonl(path: str, limit: Optional[int] = None) -> List[Dict]:
    with open(path) as f:
        recs = [json.loads(line) for line in f if line.strip()]
    return recs[:limit] if limit else recs

def _instance_from_record(rec: Dict[str, Any]) -> Instance:
    h   = rec["hidden"]
    hrl = {e["rule_name"]: set(e["hidden"]) for e in h.get("hidden_fields", [])}
    rules = []
    for r in h["rules"]:
        hides = hrl.get(r["name"], set())
        t, e  = r["trigger"], r["effect"]
        rules.append(Rule(
            name=r["name"],
            trigger=Trigger(kind=t["kind"], i=t["i"], j=t.get("j",-1), k=t.get("k",0),
                            hidden_kind=("trigger_kind" in hides), hidden_k=("trigger_k" in hides)),
            effect=Effect(kind=e["kind"], target=e["target"],
                          delta=e.get("delta",0), source=e.get("source",-1),
                          hidden_kind=("effect_kind" in hides), hidden_delta=("effect_delta" in hides)),
        ))
    initial = State(tuple(EntityState(phase=s["phase"], flux=s["flux"]) for s in h["initial_state"]))
    return Instance(
        id=rec["id"], seed=rec["seed"], difficulty=rec["difficulty"],
        world=World(initial=initial, rules=tuple(rules), horizon=h["horizon"]),
        public_rule_descriptions=[], hidden_fields=h.get("hidden_fields", []),
        metacog_ground_truth=h["metacog_ground_truth"],
        true_unknown_ranking=h["true_unknown_ranking"],
        oracle_objective=h.get("oracle_best"),
    )

KBENCH_RECORDS = _load_jsonl(DATA_PATH, limit=KBENCH_LIMIT)
print(f"Loaded {len(KBENCH_RECORDS)} instances.")

In [ ]:
# 5. Scoring helper
def _extract_json(text: str) -> Dict[str, Any]:
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end == -1:
        return {}
    try:
        return json.loads(text[start:end+1])
    except json.JSONDecodeError:
        return {}

def score_text_response(raw_text: str, rec: Dict[str, Any]) -> Dict[str, Any]:
    _zero = dict(composite=0.0, objective=0.0, calibration=0.0, attention=0.0, executive=0.0)
    raw = _extract_json(raw_text)
    if not raw:
        return {**_zero, "error": "no_json"}
    inst = _instance_from_record(rec)
    resp = validate_response(raw)
    bd   = score_response(resp, inst,
                          best_obj=rec["hidden"].get("oracle_best"),
                          worst_obj=rec["hidden"].get("oracle_worst"))
    return bd.to_dict()

In [ ]:
# 6. CIPHER benchmark task
# Kaggle injects `llm` automatically — no need to construct GoogleGenAI or
# any model client. Just call llm.prompt() and Kaggle handles the rest.
import kaggle_benchmarks as kbench
from tqdm.auto import tqdm

@kbench.task(name="cipher_metacog")
def cipher_task(llm):
    """
    CIPHER — Calibrated Introspection via Partially Hidden Environment Rules.

    The LLM receives a micro-world with hidden causal rules in invented vocabulary
    and must return JSON assessing its own knowledge, ranking unknowns, and planning
    under uncertainty.

    Scored: Objective (35%) · Calibration (25%) · Attention (20%) · Executive (20%).
    """
    scores = []
    for rec in tqdm(KBENCH_RECORDS, desc="cipher", leave=False):
        try:
            reply     = llm.prompt(rec["prompt"])
            breakdown = score_text_response(str(reply), rec)
            composite = breakdown.get("composite", 0.0)
        except Exception as e:
            composite = 0.0
            breakdown = {"error": str(e)}
        scores.append(composite)
        kbench.assertions.assert_true(
            composite >= 0.5,
            f"[{rec['id']}] composite {composite:.3f} < 0.5 — {breakdown}",
        )

    mean = sum(scores) / max(len(scores), 1)
    print(f"Mean composite: {mean:.3f} over {len(scores)} instances")
    return mean

print(f"cipher_task defined — {len(KBENCH_RECORDS)} instances.")

In [ ]:
# 7. Run the benchmark
# kbench.llm is the model Kaggle injects into this Benchmark Task notebook.
# This single call runs all instances and registers results on Kaggle Benchmarks.
cipher_task.run(kbench.llm)